In [0]:
-- QUALITY FILTERS APPLIED PER SPECIES
--   ─────────────────────────────────────
--   Each species has different sensor types and quality rules:
--
--   Reef Manta Ray  → Remove manually marked outliers only
--
--   Whale Shark     → Remove outliers and keep only good Argos positions
--                     What is Argos LC?
--                     Argos is a satellite tracking system. Each position
--                     gets a quality grade called Location Class (LC):
--                       LC 3 = most accurate (error under 250 metres)
--                       LC 2 = accurate (error under 500 metres)
--                       LC 1 = acceptable (error under 1.5 km)
--                       LC 0, A, B, Z = poor quality — excluded
--                     We keep GPS positions plus LC 1, 2, 3 only.
--
--   Sperm Whale     → Remove outliers and GPS must have 4 or more satellites
--                     Fastloc GPS positions with fewer than 4 satellites
--                     are considered unreliable. Standard scientific threshold.
--
--   Green Turtle    → GPS positions only
--                     This dataset is 95% GPS. The Argos positions are
--                     mostly poor quality LC B and A so we exclude them.
--                     We lose only 4.6% of rows but gain a clean dataset.
--

In [0]:
-- REEF MANTA RAY SILVER                                    
-- 25 animals, Indonesian waters, 2014-2022  
-- What is the locality column?
-- Unique to this dataset — tells us which Indonesian study site each
-- observation came from: Raja Ampat, Komodo, Nusa Penida or Berau.
-- Very useful for comparing conditions across four different reef systems.
--
-- H3 takes LONGITUDE first then LATITUDE — not the other way around.

In [0]:
CREATE OR REPLACE TABLE mcphersonsharyn_silver.manta_ray_silver AS
SELECT
CAST(`event-id` AS BIGINT) AS event_id,
CAST(`individual-local-identifier` AS STRING) AS animal_id,
CAST(`tag-local-identifier` AS STRING) AS tag_id,
'Reef Manta Ray' AS species_common,
'Mobula alfredi' AS species_scientific,
`locality` AS locality,
`study-name` AS study_name,
`sensor-type` AS sensor_type,
TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS') AS timestamp_utc,
CAST(`location-lat` AS DOUBLE) AS latitude,
CAST(`location-long` AS DOUBLE) AS longitude,
`location-lat` AS latitude_raw,
`location-long` AS longitude_raw,
YEAR(TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS')) AS year,
MONTH(TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS')) AS month,
CASE
WHEN MONTH(TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS')) IN (12,1,2) THEN 'Winter'
WHEN MONTH(TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS')) IN (3,4,5) THEN 'Spring'
WHEN MONTH(TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS')) IN (6,7,8) THEN 'Summer'
WHEN MONTH(TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS')) IN (9,10,11) THEN 'Autumn'
END AS season,
h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 4) AS h3_res4,
h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 5) AS h3_res5,
h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 6) AS h3_res6,
NULL AS gps_satellite_count,
current_timestamp() AS processed_timestamp
FROM mcphersonsharyn_bronze.bronze_manta_ray_raw
WHERE `location-lat` IS NOT NULL
AND `location-long` IS NOT NULL
AND `timestamp` IS NOT NULL
AND `individual-local-identifier` IS NOT NULL
AND CAST(`location-lat` AS DOUBLE) BETWEEN -90 AND 90
AND CAST(`location-long` AS DOUBLE) BETWEEN -180 AND 180
AND (`manually-marked-outlier` IS NULL OR `manually-marked-outlier` = 0);

In [0]:
-- WHALE SHARK SILVER                                   
-- 41 animals, Gulf of Mexico, 2009-2015 

In [0]:
CREATE OR REPLACE TABLE mcphersonsharyn_silver.whale_shark_silver AS
SELECT
CAST(`event-id` AS BIGINT) AS event_id,
CAST(`individual-local-identifier` AS STRING) AS animal_id,
CAST(`tag-local-identifier` AS STRING) AS tag_id,
'Whale Shark' AS species_common,
'Rhincodon typus' AS species_scientific,
'Gulf of Mexico' AS locality,
`study-name` AS study_name,
`sensor-type` AS sensor_type,
TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS') AS timestamp_utc,
CAST(`location-lat` AS DOUBLE) AS latitude,
CAST(`location-long` AS DOUBLE) AS longitude,
`location-lat` AS latitude_raw,
`location-long` AS longitude_raw,
YEAR(TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS')) AS year,
MONTH(TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS')) AS month,
CASE
WHEN MONTH(TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS')) IN (12,1,2) THEN 'Winter'
WHEN MONTH(TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS')) IN (3,4,5) THEN 'Spring'
WHEN MONTH(TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS')) IN (6,7,8) THEN 'Summer'
WHEN MONTH(TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS')) IN (9,10,11) THEN 'Autumn'
END AS season,
h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 4) AS h3_res4,
h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 5) AS h3_res5,
h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 6) AS h3_res6,
NULL AS gps_satellite_count,
current_timestamp() AS processed_timestamp
FROM mcphersonsharyn_bronze.bronze_whale_shark_raw
WHERE `location-lat` IS NOT NULL
AND `location-long` IS NOT NULL
AND `timestamp` IS NOT NULL
AND `individual-local-identifier` IS NOT NULL
AND CAST(`location-lat` AS DOUBLE) BETWEEN -90 AND 90
AND CAST(`location-long` AS DOUBLE) BETWEEN -180 AND 180
AND (`algorithm-marked-outlier` IS NULL OR `algorithm-marked-outlier` = '0')
AND (`sensor-type` = 'gps' OR `argos:lc` IN ('1', '2', '3'));

In [0]:
-- SPERM WHALE SILVER                                   
-- 17 animals, Gulf of Mexico, 2011-2013, Fastloc GPS only   

In [0]:
CREATE OR REPLACE TABLE mcphersonsharyn_silver.sperm_whale_silver AS
SELECT
CAST(`event-id` AS BIGINT) AS event_id,
CAST(`individual-local-identifier` AS STRING) AS animal_id,
CAST(`tag-local-identifier` AS STRING) AS tag_id,
'Sperm Whale' AS species_common,
'Physeter macrocephalus' AS species_scientific,
'Gulf of Mexico' AS locality,
`study-name` AS study_name,
`sensor-type` AS sensor_type,
TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS') AS timestamp_utc,
CAST(`location-lat` AS DOUBLE) AS latitude,
CAST(`location-long` AS DOUBLE) AS longitude,
`location-lat` AS latitude_raw,
`location-long` AS longitude_raw,
YEAR(TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS')) AS year,
MONTH(TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS')) AS month,
CASE
WHEN MONTH(TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS')) IN (12,1,2) THEN 'Winter'
WHEN MONTH(TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS')) IN (3,4,5) THEN 'Spring'
WHEN MONTH(TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS')) IN (6,7,8) THEN 'Summer'
WHEN MONTH(TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS')) IN (9,10,11) THEN 'Autumn'
END AS season,
h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 4) AS h3_res4,
h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 5) AS h3_res5,
h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 6) AS h3_res6,
CAST(`gps:satellite-count` AS INTEGER) AS gps_satellite_count,
current_timestamp() AS processed_timestamp
FROM mcphersonsharyn_bronze.bronze_sperm_whale_raw
WHERE `location-lat` IS NOT NULL
AND `location-long` IS NOT NULL
AND `timestamp` IS NOT NULL
AND `individual-local-identifier` IS NOT NULL
AND CAST(`location-lat` AS DOUBLE) BETWEEN -90 AND 90
AND CAST(`location-long` AS DOUBLE) BETWEEN -180 AND 180
AND (`manually-marked-outlier` IS NULL OR `manually-marked-outlier` = 0)
AND (`gps:satellite-count` IS NULL OR CAST(`gps:satellite-count` AS INTEGER) >= 4);

In [0]:
-- Green Turtle Silver
-- 33 animals, Western Indian Ocean, 2012-2019, GPS only  

In [0]:
CREATE OR REPLACE TABLE mcphersonsharyn_silver.green_turtle_silver AS
SELECT
CAST(`event-id` AS BIGINT) AS event_id,
CAST(`individual-local-identifier` AS STRING) AS animal_id,
CAST(`tag-local-identifier` AS STRING) AS tag_id,
'Green Turtle' AS species_common,
'Chelonia mydas' AS species_scientific,
CAST(NULL AS STRING) AS locality,
`study-name` AS study_name,
`sensor-type` AS sensor_type,
TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS') AS timestamp_utc,
CAST(`location-lat` AS DOUBLE) AS latitude,
CAST(`location-long` AS DOUBLE) AS longitude,
`location-lat` AS latitude_raw,
`location-long` AS longitude_raw,
YEAR(TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS')) AS year,
MONTH(TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS')) AS month,
CASE
WHEN MONTH(TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS')) IN (12,1,2) THEN 'Winter'
WHEN MONTH(TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS')) IN (3,4,5) THEN 'Spring'
WHEN MONTH(TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS')) IN (6,7,8) THEN 'Summer'
WHEN MONTH(TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS')) IN (9,10,11) THEN 'Autumn'
END AS season,
h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 4) AS h3_res4,
h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 5) AS h3_res5,
h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 6) AS h3_res6,
CAST(NULL AS INT) AS gps_satellite_count,
current_timestamp() AS processed_timestamp
FROM mcphersonsharyn_bronze.bronze_green_turtle_raw
WHERE `location-lat` IS NOT NULL
AND `location-long` IS NOT NULL
AND `timestamp` IS NOT NULL
AND `individual-local-identifier` IS NOT NULL
AND CAST(`location-lat` AS DOUBLE) BETWEEN -90 AND 90
AND CAST(`location-long` AS DOUBLE) BETWEEN -180 AND 180
AND (`algorithm-marked-outlier` IS NULL OR `algorithm-marked-outlier` = '0')
AND `sensor-type` = 'gps';

In [0]:
-- UNIFIED MIGRATION VIEW (ALL 4 SPECIES)  

In [0]:
-- Combines all four species Silver tables into one view using UNION ALL.
-- The Gold layer reads from this single view — only ever joins to one table.
-- When we add new species in future we just add another UNION ALL block.
--
-- Why UNION ALL and not JOIN?
-- UNION ALL stacks rows on top of each other — same columns, more rows.
-- JOIN combines columns side by side — same rows, more columns.
-- We want one long list of all animal observations so we use UNION ALL.

In [0]:
CREATE OR REPLACE VIEW mcphersonsharyn_silver.migration_silver AS
SELECT event_id, animal_id, tag_id, species_common, species_scientific,
locality, study_name, sensor_type, timestamp_utc,
latitude, longitude, latitude_raw, longitude_raw,
year, month, season, h3_res4, h3_res5, h3_res6,
gps_satellite_count, processed_timestamp
FROM mcphersonsharyn_silver.manta_ray_silver
UNION ALL
SELECT event_id, animal_id, tag_id, species_common, species_scientific,
locality, study_name, sensor_type, timestamp_utc,
latitude, longitude, latitude_raw, longitude_raw,
year, month, season, h3_res4, h3_res5, h3_res6,
gps_satellite_count, processed_timestamp
FROM mcphersonsharyn_silver.whale_shark_silver
UNION ALL
SELECT event_id, animal_id, tag_id, species_common, species_scientific,
locality, study_name, sensor_type, timestamp_utc,
latitude, longitude, latitude_raw, longitude_raw,
year, month, season, h3_res4, h3_res5, h3_res6,
gps_satellite_count, processed_timestamp
FROM mcphersonsharyn_silver.sperm_whale_silver
UNION ALL
SELECT event_id, animal_id, tag_id, species_common, species_scientific,
locality, study_name, sensor_type, timestamp_utc,
latitude, longitude, latitude_raw, longitude_raw,
year, month, season, h3_res4, h3_res5, h3_res6,
gps_satellite_count, processed_timestamp
FROM mcphersonsharyn_silver.green_turtle_silver;

In [0]:
-- Validation

In [0]:
-- All four species — check row counts, animal counts and H3 nulls
SELECT
species_common,
count(*) AS total_rows,
count(DISTINCT animal_id) AS unique_animals,
count(*) filter(WHERE h3_res4 IS NULL) AS null_h3,
min(year) AS earliest_year,
max(year) AS latest_year
FROM mcphersonsharyn_silver.migration_silver
GROUP BY species_common
ORDER BY species_common;

In [0]:
-- Check season distribution — all four seasons should appear for each species
SELECT species_common, season, count(*) AS observations
FROM mcphersonsharyn_silver.migration_silver
GROUP BY species_common, season
ORDER BY species_common, season;

In [0]:
-- Check locality coverage — manta ray and green turtle should show named sites
SELECT species_common, locality, count(*) AS observations
FROM mcphersonsharyn_silver.migration_silver
GROUP BY species_common, locality
ORDER BY species_common, locality;

In [0]:
CREATE OR REPLACE VIEW mcphersonsharyn_silver.migration_silver AS
SELECT event_id, animal_id, tag_id, species_common, species_scientific,
CAST(locality AS STRING) AS locality,
study_name, sensor_type, timestamp_utc,
latitude, longitude, latitude_raw, longitude_raw,
year, month, season, h3_res4, h3_res5, h3_res6,
gps_satellite_count, processed_timestamp
FROM mcphersonsharyn_silver.manta_ray_silver
UNION ALL
SELECT event_id, animal_id, tag_id, species_common, species_scientific,
CAST(locality AS STRING) AS locality,
study_name, sensor_type, timestamp_utc,
latitude, longitude, latitude_raw, longitude_raw,
year, month, season, h3_res4, h3_res5, h3_res6,
gps_satellite_count, processed_timestamp
FROM mcphersonsharyn_silver.whale_shark_silver
UNION ALL
SELECT event_id, animal_id, tag_id, species_common, species_scientific,
CAST(locality AS STRING) AS locality,
study_name, sensor_type, timestamp_utc,
latitude, longitude, latitude_raw, longitude_raw,
year, month, season, h3_res4, h3_res5, h3_res6,
gps_satellite_count, processed_timestamp
FROM mcphersonsharyn_silver.sperm_whale_silver
UNION ALL
SELECT event_id, animal_id, tag_id, species_common, species_scientific,
CAST(locality AS STRING) AS locality,
study_name, sensor_type, timestamp_utc,
latitude, longitude, latitude_raw, longitude_raw,
year, month, season, h3_res4, h3_res5, h3_res6,
gps_satellite_count, processed_timestamp
FROM mcphersonsharyn_silver.green_turtle_silver;

In [0]:
-- Check locality coverage — manta ray and green turtle should show named sites
SELECT species_common, locality, count(*) AS observations
FROM mcphersonsharyn_silver.migration_silver
GROUP BY species_common, locality
ORDER BY species_common, locality;

In [0]:
-- Check which tables have a locality column
SELECT 'manta_ray' AS species, locality FROM mcphersonsharyn_silver.manta_ray_silver LIMIT 1;
SELECT 'whale_shark' AS species, locality FROM mcphersonsharyn_silver.whale_shark_silver LIMIT 1;
SELECT 'sperm_whale' AS species, locality FROM mcphersonsharyn_silver.sperm_whale_silver LIMIT 1;
SELECT 'green_turtle' AS species, locality FROM mcphersonsharyn_silver.green_turtle_silver LIMIT 1;

In [0]:
-- Run these one at a time
SELECT locality FROM mcphersonsharyn_silver.manta_ray_silver LIMIT 1;

In [0]:
SELECT locality FROM mcphersonsharyn_silver.whale_shark_silver LIMIT 1;


In [0]:
SELECT locality FROM mcphersonsharyn_silver.sperm_whale_silver LIMIT 1;

In [0]:
SELECT locality FROM mcphersonsharyn_silver.green_turtle_silver LIMIT 1;

In [0]:
CREATE OR REPLACE TABLE mcphersonsharyn_silver.green_turtle_silver AS
SELECT
CAST(`event-id` AS BIGINT) AS event_id,
CAST(`individual-local-identifier` AS STRING) AS animal_id,
CAST(`tag-local-identifier` AS STRING) AS tag_id,
'Green Turtle' AS species_common,
'Chelonia mydas' AS species_scientific,
CAST(NULL AS STRING) AS locality,
`study-name` AS study_name,
`sensor-type` AS sensor_type,
TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS') AS timestamp_utc,
CAST(`location-lat` AS DOUBLE) AS latitude,
CAST(`location-long` AS DOUBLE) AS longitude,
`location-lat` AS latitude_raw,
`location-long` AS longitude_raw,
YEAR(TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS')) AS year,
MONTH(TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS')) AS month,
CASE
WHEN MONTH(TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS')) IN (12,1,2) THEN 'Winter'
WHEN MONTH(TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS')) IN (3,4,5) THEN 'Spring'
WHEN MONTH(TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS')) IN (6,7,8) THEN 'Summer'
WHEN MONTH(TO_TIMESTAMP(`timestamp`, 'yyyy-MM-dd HH:mm:ss.SSS')) IN (9,10,11) THEN 'Autumn'
END AS season,
h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 4) AS h3_res4,
h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 5) AS h3_res5,
h3_longlatash3(CAST(`location-long` AS DOUBLE), CAST(`location-lat` AS DOUBLE), 6) AS h3_res6,
CAST(NULL AS INTEGER) AS gps_satellite_count,
current_timestamp() AS processed_timestamp
FROM mcphersonsharyn_bronze.bronze_green_turtle_raw
WHERE `location-lat` IS NOT NULL
AND `location-long` IS NOT NULL
AND `timestamp` IS NOT NULL
AND `individual-local-identifier` IS NOT NULL
AND CAST(`location-lat` AS DOUBLE) BETWEEN -90 AND 90
AND CAST(`location-long` AS DOUBLE) BETWEEN -180 AND 180
AND (`algorithm-marked-outlier` IS NULL OR `algorithm-marked-outlier` = '0')
AND `sensor-type` = 'gps';

In [0]:
SELECT species_common, locality, count(*) AS observations
FROM mcphersonsharyn_silver.migration_silver
GROUP BY species_common, locality
ORDER BY species_common, locality;

In [0]:

--— Full Silver Layer Summary


SELECT
'SST Silver' AS table_name,
count(*) AS total_rows,
count(DISTINCT year) AS years,
count(DISTINCT month) AS months,
round(min(sea_surface_temperature), 2) AS min_value,
round(max(sea_surface_temperature), 2) AS max_value,
round(avg(sea_surface_temperature), 2) AS avg_value,
'degrees C' AS units
FROM mcphersonsharyn_silver.sst_silver

UNION ALL

SELECT
'CC Silver',
count(*),
count(DISTINCT year),
count(DISTINCT month),
round(min(chlorophyll_concentration), 4),
round(max(chlorophyll_concentration), 4),
round(avg(chlorophyll_concentration), 4),
'mg/m3'
FROM mcphersonsharyn_silver.cc_silver

UNION ALL

SELECT
'ENSO Silver',
count(*),
count(DISTINCT year),
count(DISTINCT month),
round(min(mei_value), 2),
round(max(mei_value), 2),
round(avg(mei_value), 2),
'MEI index'
FROM mcphersonsharyn_silver.enso_silver

UNION ALL

SELECT
'Microplastics Silver',
count(*),
count(DISTINCT year),
count(DISTINCT month),
round(min(concentration), 4),
round(max(concentration), 4),
round(avg(concentration), 4),
'pieces/m3'
FROM mcphersonsharyn_silver.microplastics_silver

UNION ALL

SELECT
'Environmental Conditions Silver',
count(*),
count(DISTINCT year),
count(DISTINCT month),
round(min(sea_surface_temperature), 2),
round(max(sea_surface_temperature), 2),
round(avg(sea_surface_temperature), 2),
'combined table'
FROM mcphersonsharyn_silver.environmental_conditions_silver

UNION ALL

-- Animal tracking summary
SELECT
'Reef Manta Ray Silver',
count(*),
count(DISTINCT year),
count(DISTINCT month),
NULL,
NULL,
NULL,
count(DISTINCT animal_id) || ' animals'
FROM mcphersonsharyn_silver.manta_ray_silver

UNION ALL

SELECT
'Whale Shark Silver',
count(*),
count(DISTINCT year),
count(DISTINCT month),
NULL,
NULL,
NULL,
count(DISTINCT animal_id) || ' animals'
FROM mcphersonsharyn_silver.whale_shark_silver

UNION ALL

SELECT
'Sperm Whale Silver',
count(*),
count(DISTINCT year),
count(DISTINCT month),
NULL,
NULL,
NULL,
count(DISTINCT animal_id) || ' animals'
FROM mcphersonsharyn_silver.sperm_whale_silver

UNION ALL

SELECT
'Green Turtle Silver',
count(*),
count(DISTINCT year),
count(DISTINCT month),
NULL,
NULL,
NULL,
count(DISTINCT animal_id) || ' animals'
FROM mcphersonsharyn_silver.green_turtle_silver

UNION ALL

SELECT
'Migration Silver — All Species',
count(*),
count(DISTINCT year),
count(DISTINCT month),
NULL,
NULL,
NULL,
count(DISTINCT species_common) || ' species'
FROM mcphersonsharyn_silver.migration_silver

ORDER BY table_name;

In [0]:

-- ATLAS — Silver Layer Exploration Queries

In [0]:


-- Query 1: See one anchor point in full detail
-- Pick one H3 cell, one year, one month
-- and see all four environmental variables together


SELECT
h3_res4,
year,
month,
round(sea_surface_temperature, 2) AS sst,
round(chlorophyll_concentration, 4) AS cc,
mei_value AS enso_index,
enso_phase,
round(microplastic_concentration, 4) AS microplastics,
pollution_severity
FROM mcphersonsharyn_silver.environmental_conditions_silver
WHERE year = 2014
AND month = 3
LIMIT 10;



In [0]:

-- Query 2: See a manta ray observation alongside its
-- environmental conditions at that location and time


SELECT
m.animal_id,
m.species_common,
m.locality,
m.year,
m.month,
m.season,
round(m.latitude, 4) AS animal_lat,
round(m.longitude, 4) AS animal_lon,
round(e.sea_surface_temperature, 2) AS sst,
round(e.chlorophyll_concentration, 4) AS cc,
e.enso_phase,
e.microplastic_concentration
FROM mcphersonsharyn_silver.manta_ray_silver m
LEFT JOIN mcphersonsharyn_silver.environmental_conditions_silver e
ON m.h3_res4 = e.h3_res4
AND m.year = e.year
AND m.month = e.month
LIMIT 10;

In [0]:

-- Query 3: Monthly SST average across the study region
-- for a single year — see how temperature changes through the year



SELECT
year,
month,
count(*) AS anchor_points,
round(avg(sea_surface_temperature), 2) AS avg_sst,
round(avg(chlorophyll_concentration), 4) AS avg_cc,
enso_phase
FROM mcphersonsharyn_silver.environmental_conditions_silver
WHERE year = 2014
GROUP BY year, month, enso_phase
ORDER BY month;

In [0]:

-- Query 4: Compare SST and CC across all your years for the same month — March — to see year to year change


SELECT
year,
month,
round(avg(sea_surface_temperature), 2) AS avg_sst,
round(avg(chlorophyll_concentration), 4) AS avg_cc,
enso_phase,
count(*) AS anchor_points
FROM mcphersonsharyn_silver.environmental_conditions_silver
WHERE month = 3
GROUP BY year, month, enso_phase
ORDER BY year;

In [0]:

-- Query 5: The full anchor point picture
-- Count how many complete anchor points you have for each year
-- and how many are missing one or more of (rows with SST + CC + ENSO all present)


SELECT
year,
count(*) AS total_anchor_points,
count(*) filter(WHERE chlorophyll_concentration IS NOT NULL) AS with_cc,
count(*) filter(WHERE enso_phase IS NOT NULL) AS with_enso,
count(*) filter(WHERE microplastic_concentration IS NOT NULL) AS with_microplastics,
count(*) filter(WHERE chlorophyll_concentration IS NOT NULL
AND enso_phase IS NOT NULL) AS fully_complete
FROM mcphersonsharyn_silver.environmental_conditions_silver
GROUP BY year
ORDER BY year;